# proteomics_analysis

## Objective
Run the standardized proteomics workflow from the template configuration files.

This notebook is the main notebook counterpart to the shared script set in `workflow/scripts/python/`. It reads project-specific settings from `workflow/00_raw_data/config/`, previews the main processing steps in-notebook, and writes outputs into the corresponding workflow step folders.


## Inputs
- `workflow/00_raw_data/config/project_manifest.yaml`
- `workflow/00_raw_data/config/sample_metadata.csv`
- `workflow/00_raw_data/config/comparisons.csv`

## Main outputs
- `workflow/01_qc_normalization/`
- `workflow/02_statistics/`
- `workflow/03_visualization/`


# Import libraries


In [ ]:
import os
import re
import sys
import json
import shutil
from pathlib import Path
from functools import reduce

import yaml
import numpy as np
import pandas as pd
import plotly
import scipy
import sklearn
import statsmodels
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns


# Add python scripts to import path


In [ ]:
cwd = Path.cwd().resolve()
candidate_dirs = [p / 'workflow' / 'scripts' / 'python' for p in [cwd, *cwd.parents]]
scripts_python_dir = next((p for p in candidate_dirs if p.exists()), None)
if scripts_python_dir is None:
    raise FileNotFoundError('Could not locate workflow/scripts/python from the current notebook working directory.')
sys.path.insert(0, str(scripts_python_dir))
for module_name in ['helpers', 'filter_and_Normalize', 'Stats', 'volcano_plotter', 'heatmap_plotter', 'pca_plotter']:
    if module_name in sys.modules:
        del sys.modules[module_name]
print(f'Using script imports from: {scripts_python_dir}')


# Import current shared functions


In [ ]:
from helpers import (
    extractAnnotation,
    renameColumns,
    get_columns_for_group,
    parse_bool,
    prepare_manifest,
    generate_sample_metadata_from_columns,
    validate_sample_metadata_against_columns,
    sample_metadata_has_usable_labels,
    generate_comparisons_from_metadata,
    validate_comparisons_against_metadata,
    build_placeholder_comparisons,
)
from filter_and_Normalize import (
    filterLowPSM,
    extractAbundanceData,
    perform_prtc_pca,
    normalize_prtc,
    uq_normalization,
    read_provider_export,
    perform_primary_normalization,
    apply_column_merge_operations,
    plot_data_distribution,
)
from Stats import perform_student_t_test, calculate_foldchange, test_shapiro_wilk_normality
from volcano_plotter import plot_volcano
from heatmap_plotter import plot_heatmap
from pca_plotter import compute_pca, plot_all_2d_pca_pairs, plot_pca_3d, plot_pls_da_with_cv

# Load project configuration


In [ ]:
notebook_dir = Path.cwd().resolve()
workflow_root = notebook_dir.parent.parent
project_root = workflow_root.parent

manifest_path = workflow_root / '00_raw_data' / 'config' / 'project_manifest.yaml'
sample_metadata_path = workflow_root / '00_raw_data' / 'config' / 'sample_metadata.csv'
comparisons_path = workflow_root / '00_raw_data' / 'config' / 'comparisons.csv'

with open(manifest_path, 'r', encoding='utf-8') as handle:
    manifest = yaml.safe_load(handle)

project_cfg, input_cfg, analysis_cfg, output_cfg, flags_cfg = prepare_manifest(manifest)

sample_metadata = pd.read_csv(sample_metadata_path) if sample_metadata_path.exists() else None
comparisons = pd.read_csv(comparisons_path) if comparisons_path.exists() else None

if sample_metadata is not None and 'source_column' in sample_metadata.columns:
    source_values = sample_metadata['source_column'].astype(str).str.strip().str.lower()
    if source_values.str.startswith('fill_in_').any():
        sample_metadata = None

if comparisons is not None and {'grouping_column', 'group1', 'group2'}.issubset(comparisons.columns):
    comparison_values = pd.concat([
        comparisons['grouping_column'].astype(str),
        comparisons['group1'].astype(str),
        comparisons['group2'].astype(str),
    ], ignore_index=True).str.strip().str.lower()
    if comparison_values.str.startswith('fill_in_').any():
        comparisons = None

data_path = project_root / input_cfg['provider_export']
qc_dir = project_root / output_cfg['qc_dir']
results_dir = project_root / output_cfg['results_dir']
plots_dir = project_root / output_cfg['plots_dir']
secondary_dir = project_root / output_cfg['secondary_dir']

results_csv_dir = results_dir / 'CSV'
plots_output_dir = plots_dir / 'output'
plots_csv_dir = plots_output_dir / 'CSV'
plots_png_dir = plots_output_dir / 'PNG'
plots_svg_dir = plots_output_dir / 'SVG'
plots_html_dir = plots_output_dir / 'HTML'

for path in [qc_dir, results_dir, plots_dir, secondary_dir, results_csv_dir, plots_output_dir, plots_csv_dir, plots_png_dir, plots_svg_dir, plots_html_dir]:
    path.mkdir(parents=True, exist_ok=True)

sample_metadata if sample_metadata is not None else pd.DataFrame()

# Validate project configuration


In [ ]:
if not data_path.exists():
    raise FileNotFoundError(f'Input data file not found: {data_path}')

print('Project:', project_cfg.get('project_id', ''))
print('Title:', project_cfg.get('project_title', ''))
print('Input data:', data_path)
print('QC dir:', qc_dir)
print('Results dir:', results_dir)
print('Plots dir:', plots_dir)
print('sample_metadata.csv exists:', sample_metadata_path.exists())
print('comparisons.csv exists:', comparisons_path.exists())

# Read data


In [ ]:
input_format = str(analysis_cfg.get('input_format', 'xlsx')).lower()
index_col = analysis_cfg.get('input_index_column', 4)

data = read_provider_export(data_path, input_format=input_format, index_col=index_col)

data.shape


# Extract annotation columns


In [ ]:
annotation = extractAnnotation(data)
annotation.head()


# Optional project-specific row filtering


In [ ]:
# Apply any project-specific row filtering here.
# Example patterns:
# - retain one organism while keeping PRTC rows
# - retain one tissue subset
# - drop known contaminant rows
#
# Example:
# keep_mask = data['Description'].str.contains('Streptococcus', case=False, na=False)
# prtc_mask = data.index.astype(str).str.startswith('PRTC-')
# data = data[keep_mask | prtc_mask].copy()

data.shape


# Filter and normalize the data


In [ ]:
psm_threshold = int(analysis_cfg.get('psm_threshold', 3))
psm_column_contains = analysis_cfg.get('psm_column_contains', 'PSM')
abundance_column_contains = analysis_cfg.get('abundance_column_contains', 'Abundances (Grouped):')
rename_regexes = analysis_cfg.get('column_rename_regexes', [r'Abundances_\(Grouped\):_', r'_Female'])

filtered_psm = filterLowPSM(data, psm_value=psm_threshold, controlColumnContains=psm_column_contains)
abundance_data = extractAbundanceData(filtered_psm, abundanceColumnContains=abundance_column_contains)
renameColumns(abundance_data, regexes=rename_regexes)
abundance_data.head()


# Review sample metadata before continuing

The notebook will show the current `sample_metadata.csv` and report whether manual metadata correction is still required before comparison generation.


In [ ]:
sample_metadata_was_created = False
metadata_ready = False

if sample_metadata_path.exists():
    sample_metadata = pd.read_csv(sample_metadata_path)
    validate_sample_metadata_against_columns(sample_metadata, abundance_data.columns.tolist())
else:
    sample_metadata = generate_sample_metadata_from_columns(abundance_data.columns.tolist())
    sample_metadata.to_csv(sample_metadata_path, index=False)
    sample_metadata_was_created = True
    print(f'Created {sample_metadata_path}')

print(f'Edit sample metadata at: {sample_metadata_path}')
print('Open that file in another editor tab if treatment/group labels need correction.')
display(sample_metadata)

if not sample_metadata_has_usable_labels(sample_metadata):
    if sample_metadata_was_created:
        print('Manual metadata correction required: the notebook created a starter sample_metadata.csv, but the raw data column names do not contain usable treatment or group labels.')
    else:
        print('Manual metadata correction required: sample_metadata.csv still does not contain usable treatment or group labels.')
    print(f'Update treatment/group values in: {sample_metadata_path}')
else:
    metadata_ready = True
    print('sample_metadata.csv contains usable treatment/group labels.')

include_series = sample_metadata['include'].apply(parse_bool)
metadata_used = sample_metadata.loc[include_series].copy()


# Review comparisons before continuing

Only validate or create `comparisons.csv` after `sample_metadata.csv` has usable treatment/group labels.


In [ ]:
comparisons_ready = False

sample_metadata = pd.read_csv(sample_metadata_path)
validate_sample_metadata_against_columns(sample_metadata, abundance_data.columns.tolist())
metadata_ready = sample_metadata_has_usable_labels(sample_metadata)

if comparisons_path.exists():
    comparisons = pd.read_csv(comparisons_path)
else:
    comparisons = None

comparisons_mode = str(input_cfg.get('comparisons_mode', 'generated')).strip().lower()
print(f'comparisons_mode: {comparisons_mode}')

if not metadata_ready:
    print('Skipping comparisons setup until sample_metadata.csv is corrected and rerun.')
    print(f'Metadata file to edit: {sample_metadata_path}')
else:
    inferred_comparisons = generate_comparisons_from_metadata(sample_metadata)

    def _normalize_comparisons(df):
        if df is None or df.empty:
            return pd.DataFrame()
        cols = ['comparison_id', 'grouping_column', 'group1', 'group2', 'use_qvalue', 'pvalue_cutoff', 'qvalue_cutoff', 'abs_log2fc_cutoff', 'enabled', 'notes']
        tmp = df[cols].copy()
        tmp['use_qvalue'] = tmp['use_qvalue'].apply(parse_bool)
        tmp['enabled'] = tmp['enabled'].apply(parse_bool)
        for col in ['pvalue_cutoff', 'qvalue_cutoff', 'abs_log2fc_cutoff']:
            tmp[col] = pd.to_numeric(tmp[col], errors='coerce')
        return tmp.sort_values(['grouping_column', 'group1', 'group2', 'comparison_id']).reset_index(drop=True)

    if inferred_comparisons.empty:
        comparisons = build_placeholder_comparisons()
        comparisons.to_csv(comparisons_path, index=False)
        print('Manual comparison definition required: comparisons.csv could not be inferred from sample_metadata.csv.')
        print(f'Edit comparisons at: {comparisons_path}')
    elif comparisons_mode == 'generated' and comparisons is None:
        comparisons = inferred_comparisons
        comparisons.to_csv(comparisons_path, index=False)
        comparisons_ready = True
        print(f'Created {comparisons_path}')
    elif comparisons is not None:
        try:
            validate_comparisons_against_metadata(comparisons, sample_metadata)
            comparisons_ready = True
            print('comparisons.csv is compatible with sample_metadata.csv.')
        except Exception as exc:
            print(f'Manual comparison correction required: {exc}')
            print(f'Edit comparisons at: {comparisons_path}')
            if comparisons_mode == 'generated':
                print('To regenerate comparisons from metadata, rerun metadata_gen.py outside the notebook after reviewing the intended comparison set.')
    else:
        comparisons = inferred_comparisons
        comparisons.to_csv(comparisons_path, index=False)
        comparisons_ready = True
        print(f'Created {comparisons_path}')

if comparisons is not None:
    display(comparisons)


# PRTC review and primary normalization


In [ ]:
prtc_mask = abundance_data.index.astype(str).str.startswith('PRTC-')
print('PRTC rows detected:', int(prtc_mask.sum()))

if str(analysis_cfg.get('normalization_primary', '')).strip().lower() in {'prtc', 'prtc_median'} and prtc_mask.any():
    perform_prtc_pca(abundance_data, prtc_mask, n_components=3, perform_log2=True)

(raw_data, normalized_data), normalization_info = perform_primary_normalization(abundance_data, analysis_cfg, flags_cfg)
merge_operations = analysis_cfg.get('post_normalization_column_merges', [])
merge_log = pd.DataFrame()
if merge_operations:
    normalized_data, merge_log = apply_column_merge_operations(normalized_data, merge_operations)

normalized_column_lookup = {str(col).strip(): col for col in normalized_data.columns}
metadata_source_keys = metadata_used['source_column'].astype(str).str.strip().tolist()
missing_columns = sorted([key for key in metadata_source_keys if key not in normalized_column_lookup])
if missing_columns:
    raise KeyError(f'Source columns listed in sample_metadata.csv were not found after normalization/column merging: {missing_columns}')

resolved_columns = [normalized_column_lookup[key] for key in metadata_source_keys]
normalized_data = normalized_data[resolved_columns].copy()
metadata_used['column_name'] = resolved_columns

plot_data_distribution(raw_data, normalized_data)
normalized_data.head()


# Export QC outputs


In [ ]:
metadata_used.to_csv(qc_dir / 'sample_metadata_used.csv', index=False)
metadata_used.to_csv(qc_dir / 'metadata.txt', index=False)
metadata_used.to_csv(plots_dir / 'metadata.txt', index=False)
annotation.to_csv(qc_dir / 'annotation.csv', index=True)
raw_data.to_csv(qc_dir / 'raw_abundance_matrix.csv', index=True)
normalized_data.to_csv(qc_dir / 'normalized_abundance_matrix.csv', index=True)
pd.DataFrame([normalization_info]).to_csv(qc_dir / 'normalization_summary.csv', index=False)
if not merge_log.empty:
    merge_log.to_csv(qc_dir / 'post_normalization_column_merges.csv', index=False)

with open(qc_dir / 'software_versions.txt', 'w', encoding='utf-8') as handle:
    handle.write(f"python\t{sys.version}\n")
    handle.write(f"numpy\t{np.__version__}\n")
    handle.write(f"pandas\t{pd.__version__}\n")
    handle.write(f"scipy\t{scipy.__version__}\n")
    handle.write(f"statsmodels\t{statsmodels.__version__}\n")
    handle.write(f"plotly\t{plotly.__version__}\n")
    handle.write(f"matplotlib\t{matplotlib.__version__}\n")
    handle.write(f"seaborn\t{sns.__version__}\n")
    handle.write(f"sklearn\t{sklearn.__version__}\n")

print('QC outputs written to', qc_dir)


# Prepare comparison loop


In [ ]:
normalized_no_prtc = normalized_data.loc[~normalized_data.index.astype(str).str.startswith('PRTC-')].copy()
comparison_rows = comparisons[comparisons['enabled'].apply(parse_bool)].copy()
comparison_rows


# Run statistics and generate plots


In [ ]:
apply_uq = parse_bool(analysis_cfg.get('apply_uq_per_comparison', True))
astral_mode = parse_bool(flags_cfg.get('astral_mode', False))
default_grouping_column = analysis_cfg.get('grouping_column', 'group')
generate_qvalue_plots = parse_bool(flags_cfg.get('generate_qvalue_plots', True))
between_group_bias_mode = analysis_cfg.get('between_group_bias_mode', 'none')

def _collapse_duplicate_index(df):
    if df.index.is_unique:
        return df.copy()
    numeric_cols = df.select_dtypes(include=[np.number, bool]).columns.tolist()
    other_cols = [col for col in df.columns if col not in numeric_cols]
    parts = []
    if numeric_cols:
        parts.append(df[numeric_cols].groupby(level=0).mean())
    if other_cols:
        parts.append(df[other_cols].groupby(level=0).first())
    collapsed = pd.concat(parts, axis=1)
    return collapsed.reindex(columns=df.columns)

annotation = _collapse_duplicate_index(annotation)
normalized_no_prtc = _collapse_duplicate_index(normalized_no_prtc)

comparison_counts = []
output_dirs = {'html': plots_html_dir, 'png': plots_png_dir, 'svg': plots_svg_dir}

for _, row in comparison_rows.iterrows():
    comparison_id = row['comparison_id']
    grouping_column = row.get('grouping_column', default_grouping_column)
    if pd.isna(grouping_column) or str(grouping_column).strip() == '':
        grouping_column = default_grouping_column
    group1 = row['group1']
    group2 = row['group2']
    p_cut = float(row['pvalue_cutoff'])
    q_cut = float(row['qvalue_cutoff'])
    fc_cut = float(row['abs_log2fc_cutoff'])

    print(f'\nProcessing {comparison_id}: {group1} vs {group2}')

    samples_group1 = get_columns_for_group(metadata_used, treatmentGroupColumnID=grouping_column, group_name=group1)
    samples_group2 = get_columns_for_group(metadata_used, treatmentGroupColumnID=grouping_column, group_name=group2)
    if not samples_group1 or not samples_group2:
        raise ValueError(f'Could not resolve samples for comparison {comparison_id}: {group1} vs {group2}')

    combined_samples = samples_group1 + samples_group2
    data_subset = normalized_no_prtc[combined_samples].copy()

    if apply_uq:
        raw_subset, data_subset = uq_normalization(data_subset)
    else:
        raw_subset = data_subset.copy()

    plot_data_distribution(raw_subset, data_subset)

    result_df_t = _collapse_duplicate_index(perform_student_t_test(data_subset, samples_group1, samples_group2, astral=astral_mode, between_group_bias_mode=between_group_bias_mode))
    result_df_shapiro = _collapse_duplicate_index(test_shapiro_wilk_normality(data_subset, samples_group1, samples_group2))
    result_df_fc = _collapse_duplicate_index(calculate_foldchange(data_subset, samples_group1, samples_group2))

    merged_result = pd.concat([result_df_t, result_df_shapiro, result_df_fc, annotation], axis=1, join='inner')
    expected_cols = {
        'studentT-testStatistic': float,
        'p-value_StudentTtest': float,
        'q-value_StudentTtest': float,
        '-log10(p-value_StudentTtest)': float,
        'p-value_shapiro-G1': float,
        'p-value_shapiro-G2': float,
        'Normality-G1': object,
        'Normality-G2': object,
        'mean_expr_group1': float,
        'mean_expr_group2': float,
        'FoldChange': float,
        'log2FoldChange': float,
    }
    for col, dtype in expected_cols.items():
        if col not in merged_result.columns:
            merged_result[col] = pd.Series(index=merged_result.index, dtype=dtype)
    merged_result = merged_result.loc[:, ~merged_result.columns.duplicated()]
    if 'p-value_StudentTtest' in merged_result.columns:
        merged_result = merged_result.sort_values('p-value_StudentTtest', ascending=True)

    comparison_filename = results_csv_dir / f'{group1}_vs_{group2}_comparison.csv'
    merged_result.to_csv(comparison_filename, index=True)

    if {'p-value_StudentTtest', 'log2FoldChange'}.issubset(merged_result.columns):
        p_sig = int(((merged_result['p-value_StudentTtest'] < p_cut) & (merged_result['log2FoldChange'].abs() > fc_cut)).sum())
    else:
        p_sig = 0
    if {'q-value_StudentTtest', 'log2FoldChange'}.issubset(merged_result.columns):
        q_sig = int(((merged_result['q-value_StudentTtest'] < q_cut) & (merged_result['log2FoldChange'].abs() > fc_cut)).sum())
    else:
        q_sig = 0
    comparison_counts.append({
        'comparison_id': comparison_id,
        'grouping_column': grouping_column,
        'group1': group1,
        'group2': group2,
        'pvalue_cutoff': p_cut,
        'qvalue_cutoff': q_cut,
        'abs_log2fc_cutoff': fc_cut,
        'significant_pvalue_hits': p_sig,
        'significant_qvalue_hits': q_sig,
        'output_file': str(comparison_filename.relative_to(project_root)),
    })

    hover_fields = ['Feature', 'Description', 'FoldChange']
    plot_title = f'{group1} vs {group2}'
    plot_stem = f'volcano_plot_{group1}-vs-{group2}'
    plot_volcano(merged_result, columns=['p-value_StudentTtest', 'log2FoldChange'], pvalue_limit=p_cut, fold_change_limit=fc_cut, hoverDataList=hover_fields, plotTitle=plot_title, outFileName=plot_stem, output_dirs=output_dirs)
    if generate_qvalue_plots:
        plot_volcano(merged_result, columns=['q-value_StudentTtest', 'log2FoldChange'], pvalue_limit=q_cut, fold_change_limit=fc_cut, hoverDataList=hover_fields, plotTitle=plot_title, outFileName=plot_stem, q_values=True, output_dirs=output_dirs)

    plot_heatmap(merged_result, columns=['p-value_StudentTtest', 'log2FoldChange'], pvalueLimit=p_cut, foldChangeLimit=fc_cut, comparison=(group1, group2), samples_group1=samples_group1, samples_group2=samples_group2, output_dirs=output_dirs)
    if generate_qvalue_plots:
        plot_heatmap(merged_result, columns=['q-value_StudentTtest', 'log2FoldChange'], pvalueLimit=q_cut, foldChangeLimit=fc_cut, comparison=(group1, group2), samples_group1=samples_group1, samples_group2=samples_group2, q_values=True, output_dirs=output_dirs)

    pc_df, explained_var = compute_pca(data_subset, comparison=(group1, group2), samples_group1=samples_group1, samples_group2=samples_group2, n_components=3, scale_data=True)
    plot_all_2d_pca_pairs(pc_df, comparison=(group1, group2), explained_var=explained_var, output_dirs=output_dirs)
    plot_pca_3d(pc_df, comparison=(group1, group2), explained_var=explained_var, output_dirs=output_dirs)
    plot_pls_da_with_cv(data_subset, comparison=(group1, group2), samples_group1=samples_group1, samples_group2=samples_group2, n_components=2, scale_data=True, n_splits=5, output_dirs=output_dirs)

comparison_counts_df = pd.DataFrame(comparison_counts)
comparison_counts_df


# Write summary tables


In [ ]:
comparison_counts_df.to_csv(results_dir / 'significant_protein_counts.csv', index=False)
comparison_counts_df.to_csv(plots_csv_dir / 'significant_protein_counts.csv', index=False)

print('Results CSV dir:', results_csv_dir)
print('Plots PNG dir:', plots_png_dir)
print('Plots SVG dir:', plots_svg_dir)
print('Plots HTML dir:', plots_html_dir)


# Summary for workflow documentation


In [ ]:
print('## Number of significantly DE proteins')
print(comparison_counts_df[['comparison_id', 'significant_pvalue_hits', 'significant_qvalue_hits']].to_markdown(index=False))

print('\n## Output folders')
print(f'- QC outputs: {qc_dir.relative_to(project_root)}')
print(f'- Statistical results: {results_dir.relative_to(project_root)}')
print(f'- Visualization outputs: {plots_dir.relative_to(project_root)}')
